# AnimeLoom — Character Training & Animation Pipeline (Google Colab)

Train character LoRAs for consistent anime identity across long-form animation.
Supports **SDXL** (Animagine XL 3.1) for keyframes and **SD 1.5** (DreamShaper 8) for AnimateDiff video.

**What this notebook does:**
1. Sets up AnimeLoom environment on Colab
2. Mounts Google Drive for persistent storage
3. Downloads datasets or uploads your own character images
4. Auto-captions images with BLIP
5. Trains character LoRAs (SDXL + SD 1.5 for AnimateDiff)
6. Tests trained characters for consistency
7. Exports everything for AnimeLoom's pipeline
8. **Generates animated video** — single clips or 2+ minute long videos

| Colab Tier | GPU | Session Limit | Good For |
|------------|-----|---------------|----------|
| Free | T4 (15GB) | ~90 min | 1–2 characters, short clips |
| Pro ($10) | T4/A100 | ~12 hours | Full training + 2-min video generation |

---

## 1. Environment Setup
Clones AnimeLoom, installs dependencies, mounts Google Drive.

In [ ]:
#@title 1. Setup AnimeLoom Environment
#@markdown Clones the repo, installs deps, and mounts Google Drive.

import os, sys, subprocess
from pathlib import Path

# --- Clone AnimeLoom ---
REPO_URL = "https://github.com/JoelJohnsonThomas/AnimeLoom.git"  #@param {type:"string"}
REPO_DIR = Path("/content/AnimeLoom")

if not REPO_DIR.exists():
    print("Cloning AnimeLoom…")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"AnimeLoom already at {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Warehouse on Drive (persists across sessions) ---
WAREHOUSE = Path("/content/drive/MyDrive/AniLoom/warehouse")
WAREHOUSE.mkdir(parents=True, exist_ok=True)
os.environ["AI_CACHE_ROOT"] = str(WAREHOUSE)

for sub in ["lora", "models", "datasets/raw", "datasets/tagged", "outputs", "checkpoints"]:
    (WAREHOUSE / sub).mkdir(parents=True, exist_ok=True)

print(f"Warehouse: {WAREHOUSE}")

# --- Install dependencies ---
print("\nInstalling dependencies…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu118"],
    capture_output=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "diffusers>=0.24.0", "transformers>=4.35.0", "accelerate", "safetensors",
     "peft", "bitsandbytes", "xformers",
     "opencv-python-headless", "pillow", "numpy", "scikit-image",
     "datasets", "huggingface-hub", "tqdm",
     "controlnet-aux>=0.0.7"],
    capture_output=True,
)

import torch
print(f"\nPyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\n✅ Setup complete!")

## 2. Get Character Images
Choose **one** of the three options below.

In [ ]:
#@title 2a. Download from HuggingFace (deepghs/character_similarity)
#@markdown Downloads pre-organised anime characters with multiple images each.

SPLIT = "v0_tiny"  #@param ["v0_tiny", "v1_pruned", "v1"]
MAX_CHARACTERS = 10  #@param {type:"slider", min:5, max:200, step:5}

from scripts.prepare_dataset import download_huggingface
download_huggingface(split=SPLIT, max_characters=MAX_CHARACTERS)

print(f"\nCharacter folders in {WAREHOUSE / 'datasets/raw'}:")
for d in sorted((WAREHOUSE / 'datasets/raw').iterdir()):
    if d.is_dir():
        n = len([f for f in d.iterdir() if f.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}])
        print(f"  {d.name}: {n} images")

In [ ]:
#@title 2b. Upload your own images
#@markdown Upload images to a named character folder.

CHARACTER_NAME = "yuki"  #@param {type:"string"}

from google.colab import files
from pathlib import Path
import shutil, os

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_dir = WAREHOUSE / "datasets" / "raw" / CHARACTER_NAME.lower().replace(" ", "_")
char_dir.mkdir(parents=True, exist_ok=True)

print(f"Select 5–20 images of '{CHARACTER_NAME}' (different poses/expressions):")
uploaded = files.upload()

for fname, data in uploaded.items():
    dst = char_dir / fname
    dst.write_bytes(data)
    print(f"  Saved {fname} → {dst}")

total = len(list(char_dir.glob("*")))
print(f"\n✅ {total} images in {char_dir}")
if total < 5:
    print("⚠️  Recommend at least 5 images for good LoRA quality.")

In [ ]:
#@title 2c. Import from a Google Drive folder
#@markdown Point to a Drive folder containing character images.

CHARACTER_NAME = "yuki"  #@param {type:"string"}
DRIVE_FOLDER = "/content/drive/MyDrive/my_characters/yuki"  #@param {type:"string"}

from scripts.prepare_dataset import import_local
import_local(DRIVE_FOLDER, character_name=CHARACTER_NAME)

## 3. Auto-Caption Images with BLIP
Generates text captions for each image. Captions help the LoRA learn *what* it's seeing.

In [ ]:
#@title 3. Auto-Caption with BLIP
#@markdown Select which characters to caption (comma-separated), or "all".

CHARACTERS = "all"  #@param {type:"string"}

import os
from pathlib import Path
from scripts.prepare_dataset import caption_folder

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
raw_dir = WAREHOUSE / "datasets" / "raw"

if CHARACTERS.strip().lower() == "all":
    char_list = [d for d in sorted(raw_dir.iterdir()) if d.is_dir()]
else:
    char_list = [raw_dir / c.strip() for c in CHARACTERS.split(",")]

print(f"Captioning {len(char_list)} character(s)…\n")
for char_dir in char_list:
    if not char_dir.exists():
        print(f"⚠️  Skipping {char_dir.name} — folder not found")
        continue
    caption_folder(str(char_dir))
    print()

print("✅ Captioning complete!")

## 4. Train Character LoRA
This is the main training cell. Uses AnimeLoom's `scripts/train_lora.py` under the hood.

In [ ]:
#@title 4. Train a Single Character LoRA
#@markdown Configure training parameters and run.

CHARACTER_NAME = "yuki"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 1000  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
RESOLUTION = 1024  #@param {type:"slider", min:256, max:1024, step:128}
BASE_MODEL = "cagliostrolab/animagine-xl-3.1"  #@param ["cagliostrolab/animagine-xl-3.1", "stabilityai/stable-diffusion-xl-base-1.0", "sd2-community/stable-diffusion-2-1"]

import os, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

if USE_CAPTIONS:
    image_dir = WAREHOUSE / "datasets" / "tagged" / char_id
else:
    image_dir = WAREHOUSE / "datasets" / "raw" / char_id

if not image_dir.exists():
    raise FileNotFoundError(
        f"No images at {image_dir}. "
        f"Run step 2 to get images, then step 3 to caption them."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ No GPU!")
print(f"Training '{CHARACTER_NAME}' from {image_dir}")
print(f"Rank={LORA_RANK}  LR={LEARNING_RATE}  Steps={TRAIN_STEPS}  Res={RESOLUTION}")
print("=" * 60)

lora_dir = train(
    character_name=CHARACTER_NAME,
    image_dir=image_dir,
    rank=LORA_RANK,
    lr=LEARNING_RATE,
    steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE,
    resolution=RESOLUTION,
    base_model=BASE_MODEL,
    use_captions=USE_CAPTIONS,
)

print(f"\n✅ LoRA saved to: {lora_dir}")
print("This persists on Google Drive — available in future sessions.")

In [ ]:
#@title 4b. Batch Train Multiple Characters
#@markdown Train several characters sequentially.

CHARACTER_NAMES = "yuki, kenji, sakura"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 800  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
RESOLUTION = 1024  #@param {type:"slider", min:256, max:1024, step:128}
BASE_MODEL = "cagliostrolab/animagine-xl-3.1"  #@param ["cagliostrolab/animagine-xl-3.1", "stabilityai/stable-diffusion-xl-base-1.0", "sd2-community/stable-diffusion-2-1"]

import os, gc, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_list = [c.strip() for c in CHARACTER_NAMES.split(",")]

results = {}
for i, name in enumerate(char_list, 1):
    char_id = name.lower().replace(" ", "_")
    subdir = "tagged" if USE_CAPTIONS else "raw"
    image_dir = WAREHOUSE / "datasets" / subdir / char_id

    if not image_dir.exists():
        print(f"\n⚠️  [{i}/{len(char_list)}] Skipping '{name}' — no images at {image_dir}")
        results[name] = "SKIPPED"
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{len(char_list)}] Training '{name}'…")
    print(f"{'='*60}")

    try:
        lora_dir = train(
            character_name=name,
            image_dir=image_dir,
            rank=LORA_RANK,
            lr=LEARNING_RATE,
            steps=TRAIN_STEPS,
            batch_size=BATCH_SIZE,
            resolution=RESOLUTION,
            base_model=BASE_MODEL,
            use_captions=USE_CAPTIONS,
        )
        results[name] = f"OK → {lora_dir}"
    except Exception as e:
        print(f"❌ Failed: {e}")
        results[name] = f"FAILED: {e}"

    # Free VRAM between characters
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Batch training results:")
for name, status in results.items():
    print(f"  {name}: {status}")

## 4c. Train SD 1.5 LoRA for AnimateDiff
Your SDXL LoRAs won't work with AnimateDiff (SD 1.5). Train a separate SD 1.5 LoRA using **DreamShaper 8** as the base model. This is required for animated video generation.

In [ ]:
#@title 4c. Train SD 1.5 LoRA for AnimateDiff
#@markdown Trains a separate SD 1.5 LoRA using DreamShaper 8 (anime SD 1.5 model).
#@markdown Required for AnimateDiff video generation. Saves to `warehouse/lora/{name}_sd15/`.

CHARACTER_NAMES = "yuki_nagato, denji, sakura_haruno"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 800  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
BASE_MODEL = "Lykon/dreamshaper-8"  #@param ["Lykon/dreamshaper-8", "Lykon/dreamshaper-7", "runwayml/stable-diffusion-v1-5"]

import os, gc, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_list = [c.strip() for c in CHARACTER_NAMES.split(",")]

print(f"Training SD 1.5 LoRAs for AnimateDiff compatibility")
print(f"Base model: {BASE_MODEL}")
print(f"Resolution: 512 (auto-set for SD 1.5)")
print(f"Characters: {char_list}")
print("=" * 60)

results = {}
for i, name in enumerate(char_list, 1):
    char_id = name.lower().replace(" ", "_")
    subdir = "tagged" if USE_CAPTIONS else "raw"
    image_dir = WAREHOUSE / "datasets" / subdir / char_id

    if not image_dir.exists():
        print(f"\n⚠️  [{i}/{len(char_list)}] Skipping '{name}' — no images at {image_dir}")
        results[name] = "SKIPPED"
        continue

    # Check if SD 1.5 LoRA already exists
    sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
    if (sd15_dir / "adapter_model.safetensors").exists() or (sd15_dir / "pytorch_lora_weights.safetensors").exists():
        print(f"\n[{i}/{len(char_list)}] '{name}' already has SD 1.5 LoRA at {sd15_dir} — skipping")
        results[name] = f"EXISTS → {sd15_dir}"
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{len(char_list)}] Training SD 1.5 LoRA for '{name}'…")
    print(f"{'='*60}")

    try:
        lora_dir = train(
            character_name=name,
            image_dir=image_dir,
            rank=LORA_RANK,
            lr=LEARNING_RATE,
            steps=TRAIN_STEPS,
            batch_size=BATCH_SIZE,
            resolution=512,
            base_model=BASE_MODEL,
            use_captions=USE_CAPTIONS,
        )
        results[name] = f"OK → {lora_dir}"

        # Register SD 1.5 LoRA in memory bank
        from director.memory_bank import AssetMemoryBank
        memory = AssetMemoryBank(str(WAREHOUSE))
        lora_path = lora_dir / "pytorch_lora_weights.safetensors"
        if not lora_path.exists():
            lora_path = lora_dir / "adapter_model.safetensors"
        memory.update_character_lora(name, str(lora_path), model_type="sd15")
        print(f"  Registered SD 1.5 LoRA in memory bank")

    except Exception as e:
        print(f"❌ Failed: {e}")
        results[name] = f"FAILED: {e}"

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("SD 1.5 LoRA training results:")
for name, status in results.items():
    print(f"  {name}: {status}")
print(f"\nSD 1.5 LoRAs saved to: warehouse/lora/{{name}}_sd15/")

## 5. Test Trained Character
Generate sample images with the trained LoRA to verify consistency.

In [ ]:
#@title 5. Test Character LoRA
#@markdown Generate images to verify the trained character looks consistent.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
TEST_PROMPT = "yuki_nagato, 1girl, smiling, looking at viewer, anime portrait, detailed, masterpiece"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality"  #@param {type:"string"}
NUM_IMAGES = 4  #@param {type:"slider", min:1, max:8, step:1}
GUIDANCE_SCALE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
INFERENCE_STEPS = 30  #@param {type:"slider", min:10, max:50, step:5}
IMAGE_SIZE = 768  #@param {type:"slider", min:512, max:1024, step:128}

import os, json, gc, torch
from pathlib import Path
from diffusers import DPMSolverMultistepScheduler
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")
lora_dir = WAREHOUSE / "lora" / char_id

# Load metadata
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No trained LoRA for '{CHARACTER_NAME}' at {lora_dir}")

meta = json.loads(meta_path.read_text())
is_sdxl = meta.get("is_sdxl", "xl" in meta.get("base_model", "").lower() or "animagine" in meta.get("base_model", "").lower())

print(f"Character: {meta['character_name']}")
print(f"Base model: {meta['base_model']} ({'SDXL' if is_sdxl else 'SD'})")
print(f"Rank: {meta['rank']}  Steps: {meta['training_steps']}  Images: {meta['num_images']}")

# Free VRAM from any prior work
gc.collect()
torch.cuda.empty_cache()

# Load pipeline
print("\nLoading model…")
if is_sdxl:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
    )
else:
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
        safety_checker=None,
    )

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")

# Save VRAM on T4 — decode large images in slices/tiles
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

# Load LoRA via PEFT (matches how train_lora.py saves)
from peft import PeftModel
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()
print("LoRA loaded via PEFT.")

# Generate
print(f"\nGenerating {NUM_IMAGES} images at {IMAGE_SIZE}x{IMAGE_SIZE}…")
images = []
for i in range(NUM_IMAGES):
    with torch.autocast("cuda"):
        result = pipe(
            TEST_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_SIZE,
            width=IMAGE_SIZE,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=torch.Generator("cuda").manual_seed(42 + i),
        )
    images.append(result.images[0])

# Display
cols = min(NUM_IMAGES, 4)
rows = (NUM_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if NUM_IMAGES == 1:
    axes = [axes]
else:
    axes = axes.flat if hasattr(axes, 'flat') else [axes]

for i, (ax, img) in enumerate(zip(axes, images)):
    ax.imshow(img)
    ax.set_title(f"Seed {42+i}")
    ax.axis("off")

# Hide empty subplots
for j in range(i + 1, len(list(axes)) if hasattr(axes, '__len__') else 0):
    axes[j].axis("off")

plt.suptitle(f"'{CHARACTER_NAME}' — consistency test", fontsize=14)
plt.tight_layout()
plt.show()

# Save test outputs
test_dir = WAREHOUSE / "outputs" / f"test_{char_id}"
test_dir.mkdir(parents=True, exist_ok=True)
for i, img in enumerate(images):
    img.save(test_dir / f"test_{i+1}.png")
print(f"\nSaved to {test_dir}")

## 6. Register Characters in AssetMemoryBank
Makes trained characters available to AnimeLoom's Director Agent.

In [ ]:
#@title 6. Register Characters in Memory Bank
#@markdown Registers all tagged characters and links their trained LoRAs.

import os, json
from pathlib import Path
from director.memory_bank import AssetMemoryBank

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
memory = AssetMemoryBank(str(WAREHOUSE))

tagged_dir = WAREHOUSE / "datasets" / "tagged"
lora_root = WAREHOUSE / "lora"

if not tagged_dir.exists():
    print("No tagged datasets. Run step 3 first.")
else:
    for char_dir in sorted(tagged_dir.iterdir()):
        if not char_dir.is_dir():
            continue

        name = char_dir.name
        images = sorted(
            str(f) for f in char_dir.iterdir()
            if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        )
        if not images:
            continue

        # Read description from first caption
        desc = ""
        txt = next(char_dir.glob("*.txt"), None)
        if txt:
            desc = txt.read_text().strip()

        # Create or skip
        existing = memory.get_character(name)
        if existing:
            print(f"  '{name}' already registered")
        else:
            cid = memory.create_character(name=name, images=images, description=desc)
            print(f"  Registered '{name}' ({len(images)} images) -> {cid}")

        # Link LoRA if trained (PEFT saves as adapter_model.safetensors)
        lora_dir = lora_root / name
        meta_path = lora_dir / "metadata.json"
        if meta_path.exists():
            # Find the actual weight file
            adapter_path = lora_dir / "adapter_model.safetensors"
            legacy_path = lora_dir / "pytorch_lora_weights.safetensors"
            if adapter_path.exists():
                lora_path = str(adapter_path)
            elif legacy_path.exists():
                lora_path = str(legacy_path)
            else:
                lora_path = str(adapter_path)  # store expected path
            memory.update_character_lora(name, lora_path)
            print(f"    Linked LoRA: {lora_path}")

    memory.save_checkpoint()
    print(f"\n✅ Memory bank saved. {len(memory.list_characters())} characters registered.")

## 7. Dashboard & Export

In [ ]:
#@title 7a. Training Dashboard
#@markdown View all trained characters and their stats.

import os, json
from pathlib import Path
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
lora_root = WAREHOUSE / "lora"

if not lora_root.exists():
    print("No trained LoRAs yet.")
else:
    characters = [d for d in sorted(lora_root.iterdir()) if d.is_dir()]
    if not characters:
        print("No trained LoRAs yet.")
    else:
        names, n_images, sizes_mb = [], [], []
        print(f"{'Character':<20} {'Images':>7} {'Steps':>7} {'Rank':>5} {'Size':>8} {'Model'}")
        print("-" * 80)

        for char_dir in characters:
            meta_path = char_dir / "metadata.json"
            if not meta_path.exists():
                continue
            m = json.loads(meta_path.read_text())

            # PEFT saves as adapter_model.safetensors, fallback to pytorch_lora_weights.safetensors
            weight_file = char_dir / "adapter_model.safetensors"
            if not weight_file.exists():
                weight_file = char_dir / "pytorch_lora_weights.safetensors"
            size = weight_file.stat().st_size / 1e6 if weight_file.exists() else 0

            names.append(m.get("character_name", char_dir.name))
            n_images.append(m.get("num_images", 0))
            sizes_mb.append(round(size, 1))

            print(
                f"{m.get('character_name', '?'):<20} "
                f"{m.get('num_images', '?'):>7} "
                f"{m.get('training_steps', '?'):>7} "
                f"{m.get('rank', '?'):>5} "
                f"{size:>7.1f}M "
                f"{m.get('base_model', '?').split('/')[-1]}"
            )

        if names:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
            ax1.barh(names, n_images, color="#4e79a7")
            ax1.set_xlabel("Training Images")
            ax1.set_title("Images per Character")
            ax2.barh(names, sizes_mb, color="#e15759")
            ax2.set_xlabel("Size (MB)")
            ax2.set_title("LoRA File Size")
            plt.tight_layout()
            plt.show()

In [ ]:
#@title 7b. Export All LoRAs as ZIP
#@markdown Creates a downloadable zip of all trained character LoRAs.

import os, json, zipfile
from pathlib import Path
from datetime import datetime

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
lora_root = WAREHOUSE / "lora"
export_dir = WAREHOUSE / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

characters = [d for d in sorted(lora_root.iterdir()) if d.is_dir()] if lora_root.exists() else []

if not characters:
    print("No trained LoRAs to export.")
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = export_dir / f"aniloom_loras_{ts}.zip"

    manifest = {"exported": datetime.now().isoformat(), "characters": []}

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for char_dir in characters:
            meta_path = char_dir / "metadata.json"
            if meta_path.exists():
                manifest["characters"].append(json.loads(meta_path.read_text()))

            for f in char_dir.iterdir():
                zf.write(f, f"lora/{char_dir.name}/{f.name}")
                print(f"  Added {char_dir.name}/{f.name}")

        # Write manifest inside zip
        zf.writestr("manifest.json", json.dumps(manifest, indent=2))

    size_mb = zip_path.stat().st_size / 1e6
    print(f"\n✅ Export: {zip_path}  ({size_mb:.1f} MB)")
    print(f"   {len(manifest['characters'])} characters packaged.")
    print("\nTo download: right-click the file in the Drive sidebar → Download")

## 8. Colab Survival Mode
Keep the session alive during long training runs.

In [ ]:
#@title 8. Enable Colab Survival (optional)
#@markdown Prevents Colab from disconnecting during long training.
#@markdown Checkpoints every 5 minutes to Drive.

import os, sys
sys.path.insert(0, "/content/AnimeLoom")

from cloud.colab_survival import ColabSurvival

survival = ColabSurvival(
    warehouse_path=os.environ["AI_CACHE_ROOT"],
    keepalive_interval=240,   # ping every 4 min
    checkpoint_interval=300,  # save every 5 min
)
survival.start()
print("Survival daemon running — keepalive 4min, checkpoint 5min.")
print("Your work is auto-saved to Google Drive.")

## 9. AnimateDiff Video Generation
Generate animated 16-frame clips using AnimateDiff + SD 1.5 + your character LoRA.
Requires SD 1.5 LoRAs trained in step 4c.

In [ ]:
#@title 9. Test AnimateDiff Video Generation (single clip)
#@markdown Generates a 16-frame animated clip using AnimateDiff + SD 1.5 LoRA.
#@markdown Requires SD 1.5 LoRAs from step 4c.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
PROMPT = "yuki_nagato, 1girl, close up, detailed face, sharp eyes, walking in school hallway, anime, masterpiece, best quality, highres"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, static, bad face, ugly, disfigured"  #@param {type:"string"}
NUM_FRAMES = 16  #@param {type:"slider", min:8, max:24, step:4}
GUIDANCE_SCALE = 9.0  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
INFERENCE_STEPS = 30  #@param {type:"slider", min:10, max:50, step:5}
BASE_MODEL = "Linaqruf/anything-v3-1"  #@param ["Linaqruf/anything-v3-1", "Lykon/dreamshaper-8", "runwayml/stable-diffusion-v1-5"]

import os, gc, torch, tempfile
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import HTML
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

# Free VRAM
gc.collect()
torch.cuda.empty_cache()

# Load AnimateDiff pipeline
print(f"Loading AnimateDiff pipeline with {BASE_MODEL}…")
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-3",
    torch_dtype=torch.float16,
)

pipe = AnimateDiffPipeline.from_pretrained(
    BASE_MODEL,
    motion_adapter=adapter,
    torch_dtype=torch.float16,
)

pipe.scheduler = DDIMScheduler.from_pretrained(
    BASE_MODEL,
    subfolder="scheduler",
    clip_sample=False,
    timestep_spacing="linspace",
    beta_schedule="linear",
    steps_offset=1,
)
pipe.to("cuda")
pipe.vae.enable_slicing()
print("AnimateDiff loaded.")

# Load SD 1.5 LoRA — convert PEFT format to diffusers format with unet. prefix
sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
lora_loaded = False
lora_file = sd15_dir / "pytorch_lora_weights.safetensors"
adapter_file = sd15_dir / "adapter_model.safetensors"

if lora_file.exists() or adapter_file.exists():
    try:
        from safetensors.torch import load_file, save_file

        src = str(lora_file) if lora_file.exists() else str(adapter_file)
        raw = load_file(src)

        # Convert PEFT keys → diffusers format:
        # "base_model.model.X.lora_A.default.weight" → "unet.X.lora_A.weight"
        converted = {}
        for k, v in raw.items():
            if "lora" not in k:
                continue
            ck = k.replace("base_model.model.", "").replace(".default.", ".")
            # Add unet. prefix so load_lora_weights recognises the target module
            if not ck.startswith("unet."):
                ck = "unet." + ck
            converted[ck] = v

        if converted:
            tmp_dir = Path(tempfile.mkdtemp())
            save_file(converted, str(tmp_dir / "pytorch_lora_weights.safetensors"))
            pipe.load_lora_weights(str(tmp_dir), adapter_name=char_id)
            pipe.set_adapters([char_id], adapter_weights=[0.85])
            lora_loaded = True
            print(f"Loaded SD 1.5 LoRA: {len(converted)} tensors from {sd15_dir}")
        else:
            print("No LoRA keys found in weights file")
    except Exception as e:
        print(f"LoRA load failed: {e}")
        import traceback; traceback.print_exc()
        print("Generating without LoRA…")

if not lora_loaded:
    print(f"No SD 1.5 LoRA applied — run step 4c first for character identity")

# Generate
print(f"\nGenerating {NUM_FRAMES}-frame clip…")
with torch.autocast("cuda"):
    result = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_frames=NUM_FRAMES,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=INFERENCE_STEPS,
        generator=torch.Generator("cuda").manual_seed(42),
    )

frames = result.frames[0]
print(f"Generated {len(frames)} frames")

# Save as video
output_dir = WAREHOUSE / "outputs" / "animatediff_test"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"{char_id}_test.mp4"

try:
    import cv2
    h, w = np.array(frames[0]).shape[:2]
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), 8, (w, h))
    for frame in frames:
        arr = np.array(frame)
        writer.write(cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))
    writer.release()
    print(f"Video saved: {video_path}")
except ImportError:
    print("opencv not available — saving frames as PNGs instead")
    for i, frame in enumerate(frames):
        frame.save(output_dir / f"frame_{i:03d}.png")

# Display frames
cols = min(8, len(frames))
rows = (len(frames) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows))
axes = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f"F{i}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"AnimateDiff: '{CHARACTER_NAME}'", fontsize=12)
plt.tight_layout()
plt.show()

# Clean up
del pipe, adapter
gc.collect()
torch.cuda.empty_cache()
print("\nPipeline unloaded. VRAM freed.")

## 10. Generate 2-Minute Long Video
Stitches multiple 16-frame AnimateDiff clips with cross-fade overlap to create a continuous 2+ minute animation. Uses AnimeLoom's `generate_long_video()` pipeline.

In [ ]:
#@title 10. Generate 2-Minute Long Video
#@markdown Stitches ~60 AnimateDiff clips (16 frames each, 4-frame overlap) into a 2+ minute video.
#@markdown Uses your SD 1.5 LoRA for character consistency. ~10 min on T4.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
TARGET_SECONDS = 120  #@param {type:"slider", min:30, max:300, step:30}
FPS = 8  #@param {type:"slider", min:4, max:16, step:4}
OVERLAP_FRAMES = 4  #@param {type:"slider", min:2, max:8, step:2}
GUIDANCE_SCALE = 9.0  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
INFERENCE_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}
BASE_MODEL = "Linaqruf/anything-v3-1"  #@param ["Linaqruf/anything-v3-1", "Lykon/dreamshaper-8", "runwayml/stable-diffusion-v1-5"]

#@markdown Scene prompts (one per line). The pipeline cycles through them.
SCENE_PROMPTS = """yuki_nagato, 1girl, close up, detailed face, walking through school hallway, anime, masterpiece, best quality
yuki_nagato, 1girl, close up, detailed face, sitting at desk reading book, classroom, anime, masterpiece, best quality
yuki_nagato, 1girl, close up, detailed face, standing by window, sunset light, anime, masterpiece, best quality
yuki_nagato, 1girl, close up, detailed face, walking outside, cherry blossoms, anime, masterpiece, best quality"""  #@param {type:"string"}

NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, static, ugly, bad face, disfigured"  #@param {type:"string"}

import os, gc, time, torch, tempfile
import numpy as np
from pathlib import Path

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

gc.collect()
torch.cuda.empty_cache()

# ---- Calculate clips needed ----
frames_per_clip = 16
effective_frames_per_clip = frames_per_clip - OVERLAP_FRAMES
total_frames_needed = TARGET_SECONDS * FPS
num_clips = max(1, (total_frames_needed - OVERLAP_FRAMES) // effective_frames_per_clip + 1)

print(f"Target: {TARGET_SECONDS}s at {FPS}fps = {total_frames_needed} frames")
print(f"Clips needed: {num_clips} ({frames_per_clip}f each, {OVERLAP_FRAMES}f overlap)")
print(f"Estimated time: {num_clips * 15 // 60}–{num_clips * 25 // 60} minutes on T4")
print("=" * 60)

# ---- Load AnimateDiff ----
print(f"\nLoading AnimateDiff pipeline with {BASE_MODEL}…")
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-3",
    torch_dtype=torch.float16,
)
pipe = AnimateDiffPipeline.from_pretrained(
    BASE_MODEL,
    motion_adapter=adapter,
    torch_dtype=torch.float16,
)
pipe.scheduler = DDIMScheduler.from_pretrained(
    BASE_MODEL,
    subfolder="scheduler",
    clip_sample=False,
    timestep_spacing="linspace",
    beta_schedule="linear",
    steps_offset=1,
)
pipe.to("cuda")
pipe.vae.enable_slicing()

# ---- Load SD 1.5 LoRA (PEFT → diffusers format with unet. prefix) ----
sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
lora_file = sd15_dir / "pytorch_lora_weights.safetensors"
adapter_file = sd15_dir / "adapter_model.safetensors"

if lora_file.exists() or adapter_file.exists():
    try:
        from safetensors.torch import load_file, save_file
        src = str(lora_file) if lora_file.exists() else str(adapter_file)
        raw = load_file(src)
        converted = {}
        for k, v in raw.items():
            if "lora" not in k:
                continue
            ck = k.replace("base_model.model.", "").replace(".default.", ".")
            if not ck.startswith("unet."):
                ck = "unet." + ck
            converted[ck] = v
        if converted:
            tmp_dir = Path(tempfile.mkdtemp())
            save_file(converted, str(tmp_dir / "pytorch_lora_weights.safetensors"))
            pipe.load_lora_weights(str(tmp_dir), adapter_name=char_id)
            pipe.set_adapters([char_id], adapter_weights=[0.85])
            print(f"Loaded SD 1.5 LoRA: {len(converted)} tensors from {sd15_dir}")
    except Exception as e:
        print(f"LoRA load warning: {e}")

# ---- Parse scene prompts ----
scenes = [s.strip() for s in SCENE_PROMPTS.strip().split("\n") if s.strip()]
print(f"Scene prompts: {len(scenes)}")

# ---- Generate clips ----
all_frames = []
start_time = time.time()

for clip_idx in range(num_clips):
    prompt = scenes[clip_idx % len(scenes)]
    seed = 42 + clip_idx * 7

    with torch.autocast("cuda"):
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_frames=frames_per_clip,
            guidance_scale=GUIDANCE_SCALE,
            num_inference_steps=INFERENCE_STEPS,
            generator=torch.Generator("cuda").manual_seed(seed),
        )

    clip_frames = [np.array(f) for f in result.frames[0]]

    if clip_idx == 0:
        all_frames.extend(clip_frames)
    else:
        for i in range(OVERLAP_FRAMES):
            alpha = (i + 1) / (OVERLAP_FRAMES + 1)
            prev = all_frames[-(OVERLAP_FRAMES - i)].astype(np.float32)
            curr = clip_frames[i].astype(np.float32)
            blended = ((1 - alpha) * prev + alpha * curr).clip(0, 255).astype(np.uint8)
            all_frames[-(OVERLAP_FRAMES - i)] = blended
        all_frames.extend(clip_frames[OVERLAP_FRAMES:])

    elapsed = time.time() - start_time
    eta = (elapsed / (clip_idx + 1)) * (num_clips - clip_idx - 1)
    print(f"  Clip {clip_idx+1}/{num_clips} done  "
          f"({len(all_frames)} frames, {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

    if clip_idx % 10 == 9:
        gc.collect()
        torch.cuda.empty_cache()

# ---- Save video ----
output_dir = WAREHOUSE / "outputs" / "long_video"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"{char_id}_{TARGET_SECONDS}s.mp4"

try:
    import cv2
    h, w = all_frames[0].shape[:2]
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
    for frame in all_frames:
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    writer.release()
    print(f"\nVideo saved: {video_path}")
    print(f"Duration: {len(all_frames)/FPS:.1f}s  ({len(all_frames)} frames at {FPS}fps)")
    print(f"File size: {video_path.stat().st_size / 1e6:.1f} MB")
except ImportError:
    from PIL import Image
    print("opencv not available — saving frames as PNGs")
    for i, frame in enumerate(all_frames):
        Image.fromarray(frame).save(output_dir / f"frame_{i:04d}.png")

total_time = time.time() - start_time
print(f"\nTotal generation time: {total_time/60:.1f} minutes")

del pipe, adapter
gc.collect()
torch.cuda.empty_cache()
print("Pipeline unloaded. VRAM freed.")

---

## Quick Reference

| Step | What | Time (Free/Pro) |
|------|------|------------------|
| 1 | Setup | 2 min |
| 2 | Get images | 1–5 min |
| 3 | Caption | 5–15 min |
| 4 | Train SDXL LoRA | 30–60 min / 15–30 min |
| 4c | Train SD 1.5 LoRA | 20–40 min / 10–20 min |
| 5 | Test | 1–2 min |
| 6 | Register | < 1 min |
| 7 | Dashboard + Export | < 1 min |
| 9 | AnimateDiff test clip | 1–2 min |
| 10 | 2-min long video | 10–20 min |

### After training
Your LoRAs persist on Google Drive at:
```
MyDrive/AniLoom/warehouse/lora/
├── {character_name}/                    # SDXL LoRA
│   ├── pytorch_lora_weights.safetensors
│   ├── adapter_config.json
│   └── metadata.json
└── {character_name}_sd15/               # SD 1.5 LoRA (for AnimateDiff)
    ├── pytorch_lora_weights.safetensors
    ├── adapter_config.json
    └── metadata.json
```

### Pipeline order for full animation
1. Train SDXL LoRA (step 4) — for still images / keyframes
2. Train SD 1.5 LoRA (step 4c) — for AnimateDiff video
3. Register characters (step 6)
4. Test single clip (step 9) — verify quality
5. Generate long video (step 10) — 2+ minutes

To use in AnimeLoom locally:
```bash
export AI_CACHE_ROOT=./warehouse
python main.py --script my_story.txt
```